In [0]:
source_path='/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142225_132704_0.json', name='fraud_watchlist_20261016_142225_132704_0.json', size=365, modificationTime=1792160547000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142231_461598_1.json', name='fraud_watchlist_20261016_142231_461598_1.json', size=379, modificationTime=1792160552000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142237_077304_2.json', name='fraud_watchlist_20261016_142237_077304_2.json', size=380, modificationTime=1792160558000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142242_650074_3.json', name='fraud_watchlist_20261016_142242_650074_3.json', size=383, modificationTime=1792160564000)]

In [0]:
input_stream=(spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("cloudFiles.schemaLocation", "/Volumes/finguard/source/fraud_watchlist/schema/")
              .option("cloudFiles.inferColumnTypes","true")
              .load(source_path)
)


In [0]:
from pyspark.sql import functions as F
tranformed_df=input_stream.select(
"*",
F.col("_metadata.file_path").alias("file_path"),
F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
streaming_query=(tranformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.fraud_watchlist_batch_test")
)

In [0]:
%sql
select * from finguard.bronze.fraud_watchlist_batch_test

action,city,country,effective_from,entity_id,reason_code,reason_description,reported_by,reported_source,risk_level,status,watch_type,watchlist_id,_rescued_data,file_path,ingestion_timestamp
ADD,Chennai,India,18-Jun-2026 09:04:00,4717327980595032,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,high,ACTIVE,CARD,wl000004,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142242_650074_3.json,2026-10-16T14:34:17.503Z
ADD,Delhi,India,18-Jun-2026 09:02:00,5329839542961842,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,low,ACTIVE,CARD,wl000003,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142237_077304_2.json,2026-10-16T14:34:17.503Z
ADD,Pune,India,18-Jun-2026 09:01:00,4413337312552355,BLACKLISTED_MERCHANT,Auto-generated fraud watchlist entry,SOC,External Feed,high,ACTIVE,CARD,wl000002,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142231_461598_1.json,2026-10-16T14:34:17.503Z
ADD,Hyderabad,India,18-Jun-2026 09:00:00,5008514036965665,PHISHING,Auto-generated fraud watchlist entry,SOC,Manual,high,ACTIVE,CARD,wl000001,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261016_142225_132704_0.json,2026-10-16T14:34:17.503Z


In [0]:
%sql
select * from finguard.bronze.fraud_watchlist